# Phase 2 - Section 1: Database Implementation & Data Querying

**Team Members:** Shayan Maleki (810102515), Arash Taherifardi (810102474), Ali Khosravi  
**Course:** Introduction to Data Science  

---

### Objective
This notebook serves as the formal documentation and execution environment for Section 1 of Phase 2. The primary objective is to migrate our raw GitHub herd-behavior dataset from flat CSV files into a structured, relational SQLite database. This ensures our data is professionally formatted, queryable, and ready to be integrated into the automated AI pipeline in Section 3.

### 1. Database Schema and Relational Mapping
To maintain data integrity and avoid redundancy, we are mapping our enriched dataset into six distinct, relational tables. We intentionally avoid merging features at this stage to keep the database as a clean foundational layer. The `repositories` table acts as the central hub, with all event and time-series tables referencing it via a Foreign Key.

* **`repositories`** (Main repository metadata)
    * **Primary Key:** `repo_id`
    * **Columns:** `full_name`, `owner`, `repo_name`, `language`, `topics`, `stars_count`, `forks_count`, `created_at`, `updated_at`, `pushed_at`, `license`

* **`stars`** (Raw user-repository star interactions)
    * **Logical Primary Key:** (`repo_id`, `user_id`)
    * **Foreign Key:** `repo_id` → `repositories.repo_id`
    * **Columns:** `user_id`, `username`, `starred_at`

* **`user_repo_interactions`** (Recommendation-system interaction matrix)
    * **Logical Primary Key:** (`user_id`, `repo_id`)
    * **Foreign Key:** `repo_id` → `repositories.repo_id`
    * **Columns:** `interaction`, `starred_at`

* **`daily_timeseries`** (Granular daily star-growth tracking)
    * **Logical Primary Key:** (`repo_id`, `date`)
    * **Foreign Key:** `repo_id` → `repositories.repo_id`
    * **Columns:** `daily_new_stars`, `cumulative_stars`, `daily_growth_rate`

* **`weekly_timeseries`** (Aggregated weekly star-growth tracking)
    * **Logical Primary Key:** (`repo_id`, `week`)
    * **Foreign Key:** `repo_id` → `repositories.repo_id`
    * **Columns:** `weekly_new_stars`, `cumulative_stars`, `previous_week_stars`, `weekly_growth_rate`, `herd_momentum_score`

* **`herd_modeling`** (Final target table for herd-behavior prediction)
    * **Foreign Key:** `repo_id` → `repositories.repo_id`
    * **Columns:** `early_4week_stars`, `early_avg_weekly_stars`, `early_max_weekly_stars`, `later_stars`, `later_avg_weekly_stars`, `became_high_growth`

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path


PROJECT_ROOT = Path.cwd() 
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DB_DIR = PROJECT_ROOT / "database"
DB_PATH = DB_DIR / "github_herd.db"

DB_DIR.mkdir(parents=True, exist_ok=True)

engine = create_engine(f"sqlite:///{DB_PATH}")
print(f"Database engine successfully initialized at:\n{DB_PATH}")

Database engine successfully initialized at:
c:\uni\sem6\Data-Science\Data-Science\CA-Final\database\github_herd.db


### 2. Automated Data Ingestion
With the schema defined and the database engine initialized, we now execute the data transfer. We use a mapping dictionary to strictly control which CSV files are imported, intentionally filtering out uncleaned or redundant raw files. Pandas is used to read the data into memory, and SQLAlchemy's `to_sql` method writes it directly to our local `.db` file.

In [2]:
TABLES_MAPPING = {
    "repositories_enriched.csv": "repositories",
    "stars_raw.csv": "stars",
    "user_repo_interactions.csv": "user_repo_interactions",
    "repo_star_daily_timeseries.csv": "daily_timeseries",
    "repo_star_weekly_timeseries.csv": "weekly_timeseries",
    "repo_herd_modeling_table.csv": "herd_modeling"
}

print("Initiating database ingestion sequence...\n" + "-"*45)

for file_name, table_name in TABLES_MAPPING.items():
    file_path = DATA_DIR / file_name
    
    if file_path.exists():
        print(f"Loading '{file_name}'...")
        df = pd.read_csv(file_path)
        
        df.to_sql(table_name, engine, if_exists="replace", index=False)
        print(f"  -> Inserted {len(df):,} rows into the '{table_name}' table.\n")
    else:
        print(f"  -> ERROR: '{file_name}' not found. Check the data/raw/ directory.\n")

print("-" * 45 + "\nData ingestion complete.")

Initiating database ingestion sequence...
---------------------------------------------
Loading 'repositories_enriched.csv'...
  -> Inserted 763 rows into the 'repositories' table.

Loading 'stars_raw.csv'...
  -> Inserted 216,566 rows into the 'stars' table.

Loading 'user_repo_interactions.csv'...
  -> Inserted 216,566 rows into the 'user_repo_interactions' table.

Loading 'repo_star_daily_timeseries.csv'...
  -> Inserted 11,128 rows into the 'daily_timeseries' table.

Loading 'repo_star_weekly_timeseries.csv'...
  -> Inserted 2,250 rows into the 'weekly_timeseries' table.

Loading 'repo_herd_modeling_table.csv'...
  -> Inserted 145 rows into the 'herd_modeling' table.

---------------------------------------------
Data ingestion complete.


### 3. Data Querying & Verification
To verify the integrity of the data transfer and fulfill the project's requirement for SQL execution, we run a series of `SELECT` queries against our newly populated database. 

The queries are designed to test standard aggregation and grouping logic. The results are displayed sequentially in this notebook and simultaneously exported to a text file (`outputs/sql_query_outputs.txt`) for inclusion in the final project `.zip` submission.

In [3]:
# Define the output directory and file for the final zip submission
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "sql_query_outputs.txt"

# Define verification queries
QUERIES = {
    "Total Repositories": """
        SELECT COUNT(*) AS num_repositories 
        FROM repositories;
    """,
    
    "Total Star Interactions": """
        SELECT COUNT(*) AS num_star_interactions 
        FROM stars;
    """,
    
    "Top 5 Programming Languages": """
        SELECT language, COUNT(*) AS repo_count
        FROM repositories
        GROUP BY language
        ORDER BY repo_count DESC
        LIMIT 5;
    """,
    
    "Herd Modeling Target Class Balance": """
        SELECT became_high_growth, COUNT(*) AS repo_count
        FROM herd_modeling
        GROUP BY became_high_growth;
    """
}

# Execute queries, display in notebook, and write to text file
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for title, query in QUERIES.items():
        # Execute query via pandas
        result_df = pd.read_sql_query(query, engine)
        
        # 1. Write to the text file
        f.write("=" * 50 + "\n")
        f.write(title + "\n")
        f.write("=" * 50 + "\n")
        f.write(result_df.to_string(index=False))
        f.write("\n\n")
        
        # 2. Display cleanly in the Jupyter Notebook
        print(f"\n--- {title} ---")
        display(result_df)

print(f"\nVerification complete. SQL outputs successfully saved to:\n{OUTPUT_FILE}")


--- Total Repositories ---


,num_repositories
0,763



--- Total Star Interactions ---


,num_star_interactions
0,216566



--- Top 5 Programming Languages ---


,language,repo_count
0,Python,265
1,TypeScript,129
2,JavaScript,93
3,None,70
4,Jupyter Notebook,61



--- Herd Modeling Target Class Balance ---


,became_high_growth,repo_count
0,0,72
1,1,73



Verification complete. SQL outputs successfully saved to:
c:\uni\sem6\Data-Science\Data-Science\CA-Final\outputs\sql_query_outputs.txt


---
### 4. Pipeline Execution & CI/CD Handoff

**Note :** While this notebook serves as our formal documentation and can execute the Section 1 requirements, it is not designed for the GitHub Actions CI/CD pipeline. 

To ensure seamless automation, we have mirrored the exact logic from this notebook into modular Python scripts. For pipeline execution, please ignore this notebook and run the following directly from the project root:
1. `python scripts/database_connection.py` 
2. `python scripts/import_to_db.py` (Executes the cross-platform data transfer)
3. `python scripts/run_sql_queries.py` (Generates the `outputs/sql_query_outputs.txt` artifact)